# Real vs Synthetic IMU gap — TotalCapture (common calibrated frame)

Compares **real Xsens IMU** against **synthetic IMU** (`hmodel.syn_imu.SynIMU`) in one common frame, using `hmodel.datasets.totalcapture`.

**Why the old comparison showed a huge gap**
- Raw npz poses live one global rotation from the SMPL/world frame — `prepare_motion` applies the `AMASS_ROT` root fix (skipping it leaves a uniform ~90° error on every sensor).
- Raw `RIS`/`aS` still carry the bone-to-sensor mount and the Xsens world frame, while syn `RBW`/`ACC` live in the bone/SMPL frame. `calibrate` maps real readings with the per-sequence calib files (`RMB = RIM^T @ RIS @ RSB`, `aM = RIM^T @ RIS @ aS + g`).
- Raw file quats are **wxyz** but roma uses **xyzw** — the loader converts (a silent wrong-order decode corrupts every orientation).
- Syn is mapped with the existing `imu_calibration` into the same lab frame, so ACC/GYR/ORI are all compared apples-to-apples.
- Orientation metrics use the **geodesic angle** (`orientation_error_deg`) — per-axis rotvec RMSE suffers ±π wrapping artifacts.

In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))
import torch
import roma
import numpy as np
import matplotlib.pyplot as plt

from hmodel.bmodels.smpl import SMPL
from hmodel.syn_imu import SynIMU, imu_calibration
from hmodel.datasets import totalcapture as tc

torch.set_printoptions(sci_mode=False)

# ── config ────────────────────────────────────────────────────────────
DATA_ROOT  = "D:/Projects/MemPoser/data/Total Capture"  # IMU data/ + Pose/ + IMU calib/
MODEL_PATH = "D:/HModel/models"
SUBJECT    = "s1"
ACTION     = "walking1"        # e.g. acting1, freestyle1, rom1, walking1
T0, T1     = 0, 1800             # analysis window [T0, T1)
SEED       = 0
SAVE_FIGS  = False
FIGS_DIR   = "figs_real_vs_syn"
SENSOR_SUBSET = None            # e.g. [0, 3] for Pelvis + Head only; None = all 6

SENSOR_NAMES = tc.SENSOR_NAMES
EDGE = 4                        # FD border frames to exclude from ACC metrics


In [ ]:
# ── load + calibrate real ─────────────────────────────────────────
imu = tc.parse_sensors(f"{DATA_ROOT}/IMU data/{SUBJECT}/{ACTION}_Xsens_AuxFields.sensors")
mot = tc.load_amass_npz(f"{DATA_ROOT}/Pose/{SUBJECT}/{ACTION}_poses.npz")
key = f"{SUBJECT}_{ACTION.lower()}"
rsb_all = tc.load_calib(f"{DATA_ROOT}/IMU calib/{SUBJECT}/{key}_calib_imu_bone.txt",
                         tc.bone_fix())
rim_all = tc.load_calib(f"{DATA_ROOT}/IMU calib/{SUBJECT}/{key}_calib_imu_ref.txt",
                         tc.ref_fix())
RSB_real = tc.stack_calib(rsb_all, SENSOR_NAMES)  # (6,3,3) bone->sensor
RIM = tc.stack_calib(rim_all, SENSOR_NAMES)       # (6,3,3) lab->world

# Motion prep FIRST (step + AMASS root fix), then joint window with IMU
pose_24, tran_fixed = tc.prepare_motion(mot["poses_52"], mot["trans"], mot["framerate"])
T = min(imu["aS"].shape[0], pose_24.shape[0])
sl = slice(T0, min(T1, T))
aS = tc.select_sensors(imu["aS"][sl], tc.FILE_IDX)     # (W, 6, 3) repo order
wS = tc.select_sensors(imu["wS"][sl], tc.FILE_IDX)
RIS = tc.select_sensors(imu["RIS"][sl], tc.FILE_IDX)   # (W, 6, 3, 3)
pose_aa = pose_24[sl]                                     # (W, 24, 3) SMPL frame
tran = tran_fixed[sl]                                   # (W, 3)
W = aS.shape[0]

# Real readings in the calibrated (bone/lab) frame — GlobalPose test.py convention
cal_real = tc.calibrate(aS, wS, RIS, RIM, RSB_real)
RMB_real, aM, wM = cal_real["rot_bone"], cal_real["acc"], cal_real["gyro"]
ori_real = roma.rotmat_to_rotvec(RMB_real.reshape(-1, 3, 3)).reshape(W, 6, 3)

print(f"file sensors : {imu['names']}")
print(f"window       : [{T0}, {T0 + W}) of {T} frames, gender={mot['gender']}")
print(f"mean |aS| raw = {aS.norm(dim=-1).mean():.3f}  vs  mean |aM| calib = {aM.norm(dim=-1).mean():.3f} m/s\u00b2")


In [ ]:
# ── synthesize from fixed poses, map syn into the same lab frame ──────────
smpl = SMPL(path=MODEL_PATH, gender=mot["gender"])
gen = SynIMU(smpl, fps=60)
pose_mat = roma.rotvec_to_rotmat(pose_aa.reshape(-1, 3)).reshape(W, 24, 3, 3)

torch.manual_seed(SEED)
syn_savgol = gen.syn_imu(pose_mat, tran, add_noise=False, acc_mode="savgol")
syn_paper = gen.syn_imu(pose_mat, tran, add_noise=False, acc_mode="paper")
torch.manual_seed(SEED)
syn_noisy = gen.syn_imu(pose_mat, tran, add_noise=True, acc_mode="savgol")

variants = {
    "syn clean (savgol)": syn_savgol,
    "syn clean (paper) ": syn_paper,
    "syn noisy (savgol)": syn_noisy,
}
for v in variants.values():
    c = imu_calibration(v["ACC"], v["GYR"], v["RBW"], v["RLW"], v["RLC"],
                        remove_g=True, rot_sensor_bone=v["RSB"])
    v["ACC_c"] = c["acc_calib"]
    v["GYR_c"] = c["gyro_calib"]
    v["RBW_c"] = c["rot_bone_calib"]
    v["ORI"] = roma.rotmat_to_rotvec(v["RBW_c"].reshape(-1, 3, 3)).reshape(W, 6, 3)
print("keys:", sorted(syn_savgol.keys()))

# Legacy frame error (old comparison): bone-frame syn vs sensor-frame raw
legacy = tc.orientation_error_deg(syn_savgol["RBW"], RIS).mean(dim=0)
fixed = tc.orientation_error_deg(syn_savgol["RBW_c"], RMB_real).mean(dim=0)
print(f"{'sensor':9s} | {'legacy\u00b0':>8s} {'fixed\u00b0':>8s}  (mean geodesic, old vs new frame)")
for si, sname in enumerate(SENSOR_NAMES):
    print(f"{sname:9s} | {legacy[si]:8.2f} {fixed[si]:8.2f}")


In [ ]:
# ── metrics in the common frame ────────────────────────────────
real = {"ACC": aM, "GYR": wM, "ORI": ori_real}
units = {"ACC": "m/s\u00b2", "GYR": "rad/s", "ORI": "rad"}
ssl = slice(EDGE, W - EDGE)  # exclude FD borders for ACC only

print(f"{'variant':22s} {'sensor':9s} {'mod':3s} | {'RMSE-x':>8s} {'RMSE-y':>8s} {'RMSE-z':>8s} | {'bias-x':>8s} {'bias-y':>8s} {'bias-z':>8s}")
print("-" * 100)
for vname, v in variants.items():
    for si, sname in enumerate(SENSOR_NAMES):
        for m, su in (("ACC", "ACC_c"), ("GYR", "GYR_c"), ("ORI", "ORI")):
            vv = v[su][ssl] if m == "ACC" else v[su]
            rr = real[m][ssl] if m == "ACC" else real[m]
            err = vv[:, si] - rr[:, si]
            rmse, bias = err.pow(2).mean(dim=0).sqrt(), err.mean(dim=0)
            print(f"{vname:22s} {sname:9s} {m:3s} | {rmse[0]:8.4f} {rmse[1]:8.4f} {rmse[2]:8.4f} | "
                  f"{bias[0]:+8.4f} {bias[1]:+8.4f} {bias[2]:+8.4f}  {units[m]}")
    print("-" * 100)

print("geodesic orientation error (deg) — mean / median per sensor:")
for vname, v in variants.items():
    g = tc.orientation_error_deg(v["RBW_c"], RMB_real)   # (W, 6)
    row = "  ".join(f"{s}: {g[:, si].mean():6.2f}/{g[:, si].median():6.2f}" for si, s in enumerate(SENSOR_NAMES))
    print(f"{vname:22s} {row}")


In [ ]:
# ── plots: calibrated real vs syn (savgol / paper) ──────────────────
import matplotlib
print("matplotlib", matplotlib.__version__)
if SAVE_FIGS:
    os.makedirs(os.path.join(os.getcwd(), FIGS_DIR), exist_ok=True)

sensors = range(6) if SENSOR_SUBSET is None else SENSOR_SUBSET
t = np.arange(T0, T0 + W)
traces = {"real": None, **variants}
KEY = {"ACC": "ACC_c", "GYR": "GYR_c", "ORI": "ORI"}
nfig = 0
for si in sensors:
    sname = SENSOR_NAMES[si]
    for mkey, mlabel in (("ACC", "acceleration (m/s\u00b2)"), ("GYR", "angular velocity (rad/s)"), ("ORI", "bone orientation rotvec (rad)")):
        for ax, acomp in enumerate("XYZ"):
            for tag, members in [
                ("paper", [("real", "k", 1.2), ("syn clean (paper) ", "C1", 0.9)]),
                ("savgol", [("real", "k", 1.2), ("syn clean (savgol)", "C0", 0.9), ("syn noisy (savgol)", "C3", 0.9)]),
            ]:
                fig = plt.figure(figsize=(10, 2.6))
                for vname, color, lw in members:
                    y = real[mkey][:, si, ax].numpy() if vname == "real" else traces[vname][KEY[mkey]][:, si, ax].numpy()
                    plt.plot(t, y, color=color, lw=lw, alpha=0.95, label=vname)
                plt.title(f"{sname} [{si}] \u2014 {mlabel} \u2014 {acomp}  (calibrated, real vs {tag})")
                plt.xlabel("frame"); plt.ylabel(f"{mkey}-{acomp}"); plt.legend(ncol=4, fontsize=8); plt.tight_layout()
                if SAVE_FIGS:
                    fig.savefig(os.path.join(os.getcwd(), FIGS_DIR, f"{sname}_{mkey}_{acomp}_{tag}.png"), dpi=150)
                plt.show()
                nfig += 1
print(f"done: {nfig} figures")


## Reading the gap (fixed frames)

- The **legacy vs fixed** table (cell 4) quantifies the frame fix: legacy (bone-syn vs sensor-raw) should show tens of degrees; fixed (calibrated both sides) is the true synthesis gap.
- Remaining ORI error mixes AMASS pose-fit error, mean-shape vs subject shape, and official-vs-DIP calibration differences (the pkls were built with DIP calibration; local calib files may be the official set).
- Remaining ACC error: gravity is now handled identically on both sides, so RMSE reflects motion dynamics + orientation error leaking through the `R^T(aW-g)` projection. Compare `savgol` vs `paper` traces to judge the operator, not the frame.
- Rotvec ±π jumps (sawtooth) are representation wrapping, not error — trust the geodesic table.
- Narrow the window (`T0/T1`) or `SENSOR_SUBSET` to inspect events; set `SAVE_FIGS=True` to archive PNGs.